In [6]:
import pandas as pd

file_path = "/Users/inqsoncharoen/Documents/RStudio Module/charts.csv"
df = pd.read_csv(file_path)

df.head()

,date,rank,song,artist,last-week,peak-rank,weeks-on-board
0,2021-11-06,1,Easy On Me,Adele,1.0,1,3
1,2021-11-06,2,Stay,The Kid LAROI & Justin Bieber,2.0,1,16
2,2021-11-06,3,Industry Baby,Lil Nas X & Jack Harlow,3.0,1,14
3,2021-11-06,4,Fancy Like,Walker Hayes,4.0,3,19
4,2021-11-06,5,Bad Habits,Ed Sheeran,5.0,2,18


Here, I am just trying to turn the Billboard csv (that we initially thought we'd work with, but had API problems later) into a yearly one. The original one has weekly updates to it so there are a lot of repeats. I dealt with this by giving a score, summing it, and ranking the overall one to produce a yearly chart.

In [7]:

df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

df["score"] = 101 - df["rank"]

yearly_chart = (df.groupby(["year", "song", "artist"])["score"]
                  .sum()
                  .reset_index())

yearly_chart = yearly_chart.sort_values(by=["year", "score"], ascending=[True, False])
yearly_chart[yearly_chart["year"] == 2021].head(10)


,year,song,artist,score
35484,2021,Blinding Lights,The Weeknd,3137
35713,2021,Leave The Door Open,Silk Sonic (Bruno Mars & Anderson .Paak),2942
35782,2021,Montero (Call Me By Your Name),Lil Nas X,2789
35723,2021,Levitating,Dua Lipa Featuring DaBaby,2737
35702,2021,Kiss Me More,Doja Cat Featuring SZA,2733
35783,2021,Mood,24kGoldn Featuring iann dior,2694
35845,2021,Peaches,Justin Bieber Featuring Daniel Caesar & Giveon,2595
35638,2021,Heat Waves,Glass Animals,2550
35552,2021,Drivers License,Olivia Rodrigo,2485
35537,2021,Deja Vu,Olivia Rodrigo,2466


In [9]:
output_file_path = "/Users/inqsoncharoen/Documents/RStudio Module/yearly_billboard_chart.csv"
yearly_chart.to_csv(output_file_path, index=False)

output_file_path


# d = {}
# for g, n in df_new.groupby(level=0):
#     d[g] = n.loc[:,n.sum().rank(ascending=False,method='min')<5]
    
#print(d[2018])
#print(d[2019])

'/Users/inqsoncharoen/Documents/RStudio Module/yearly_billboard_chart.csv'

In [ ]:
import requests
import base64

SPOTIPY_CLIENT_ID = "my secret!!"
SPOTIPY_CLIENT_SECRET = "my secret!!"

url = "https://accounts.spotify.com/api/token" #I used this to check my access token
headers = {
    "Authorization": "Basic " + base64.b64encode(f"{SPOTIPY_CLIENT_ID}:{SPOTIPY_CLIENT_SECRET}".encode()).decode(),
    "Content-Type": "application/x-www-form-urlencoded",
}
data = {"grant_type": "client_credentials"}

response = requests.post(url, headers=headers, data=data)

if response.status_code == 200:
    access_token = response.json().get("access_token")
    print(f"Access Token: {access_token}")
else:
    print(f"Error: {response.status_code}, Response: {response.json()}")


✅ Access Token: BQBXglqva0UyhbBTdtweID392MNJ_nEGef_JrvSGPWTEUNdnBjPlHTDsdNWKEV-EZh5nmalVTO1v4B0TENwr7jO7ynCqqDCpfC39hwewbFHIrhmg4FxUKGq47Mi9OGOy7jqH8lt3psU


I had massive issues with this part. I checked and I successfully obtained an access token, I got it to print it explicitly since I initially had issues about it. The access token part was solved. As written in the report, I spent the last two days before the (assumed) deadline figuring it out, and the problem runs deeper than just having the token -- it's also to do with Jupyter occupying its usual 8888 server. I tried forcing it onto other servers, but still didn't work. We didn't have time to work on that, nor did we have time to troubleshoot this part, that would've otherwise worked with Billboard data.

In [ ]:
import spotipy
import requests
from spotipy.oauth2 import SpotifyClientCredentials

SPOTIPY_CLIENT_ID = "572eb4f3685b49349cbd2517ac8c064e"
SPOTIPY_CLIENT_SECRET = "0e6ba40bf0e74257a939dabfb2019b9d"

url = "https://accounts.spotify.com/api/token" #MANUALLY GET IT UGHHHHHSAFJKDFJDAJ
headers = {
    "Authorization": "Basic " + base64.b64encode(f"{SPOTIPY_CLIENT_ID}:{SPOTIPY_CLIENT_SECRET}".encode()).decode(),
    "Content-Type": "application/x-www-form-urlencoded",
}
data = {"grant_type": "client_credentials"}

response = requests.post(url, headers=headers, data=data)
token = response.json().get("access_token")

sp = spotipy.Spotify(auth=token) #TRYING TO SET IT MANUALLY AHHHH AND IT STILL DOESNT WORK


In [10]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd
import time

file_path = "/Users/inqsoncharoen/Documents/RStudio Module/yearly_billboard_chart.csv"
df = pd.read_csv(file_path)

SPOTIPY_CLIENT_ID = "572eb4f3685b49349cbd2517ac8c064e"
SPOTIPY_CLIENT_SECRET = "0e6ba40bf0e74257a939dabfb2019b9d"

# Manually fetch token
client_credentials_manager = SpotifyClientCredentials(client_id=SPOTIPY_CLIENT_ID, client_secret=SPOTIPY_CLIENT_SECRET)
token = client_credentials_manager.get_access_token()

if not token:
    raise Exception("❌ Failed to get a valid token. Check your credentials.")

# Initialize Spotipy with the valid token
sp = spotipy.Spotify(auth=token)

print("✅ Successfully authenticated!")


/var/folders/hb/qtg2k1095t583b4c_cgh3d000000gn/T/ipykernel_3926/2355243210.py:14: DeprecationWarning: You're using 'as_dict = True'.get_access_token will return the token string directly in future versions. Please adjust your code accordingly, or use get_cached_token instead.
  token = client_credentials_manager.get_access_token()


✅ Successfully authenticated!


As for the cell below, I tried working on the function on my own. The get_valence() function had no issues on its own, but had 403 error (no access token issue, can't get to it on the Spotify API end). If it looks weird, it's because I got a little frustrated, and asked ChatGPT to check it and it added emojis to "check" my code and all. I have a programming background, so I know what ChatGPT was trying to do / if it was giving code consistent to my interpretation. But, it still didn't work on the Spotify end. I ran things on my computer's terminal (bash) and yeah, it wasn't  wrong syntax or that a ChatGPT correction was needed at all. The issue was much deeper.

In [ ]:
def get_valence(song, artist):
    try:
        song = song.replace(",", "").strip()
        artist = artist.replace(",", "").strip()

        query = f"{song} {artist}"
        print(f"Searching for: {query}")
        results = sp.search(q=query, type="track", limit=1)
        
        print(f"Results: {results}")

        if not results['tracks']['items']:
            print(f"⚠️ Skipping: No results for {song} by {artist}")
            return None

        track_id = results['tracks']['items'][0]['id']
        features = sp.audio_features(track_id)
        
        print(f"Features: {features}")

        if features and len(features) > 0 and 'valence' in features[0]:
            return features[0]['valence']
        else:
            print(f"⚠️ Skipping: No valence data for {song} by {artist}")
            return None
    except Exception as e:
        print(f"❌ Error retrieving valence for {song} by {artist}: {e}")
        return None


# Apply valence function to each row in DataFrame
df['valence'] = df.apply(lambda row: get_valence(row['song'], row['artist']), axis=1)

# Save updated CSV
output_file = "yearly_billboard_chart_with_valence.csv"
df.to_csv(output_file, index=False)

print(f"✅ Updated CSV saved as {output_file}")


ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Chantilly Lace Big Bopper', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': "It's All In The Game Tommy Edwards", 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Rock-in Robin Bobby Day', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported


Searching for: Chantilly Lace Big Bopper
❌ Error retrieving valence for Chantilly Lace by Big Bopper: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Chantilly+Lace+Big+Bopper&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: It's All In The Game Tommy Edwards
❌ Error retrieving valence for It's All In The Game by Tommy Edwards: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=It%27s+All+In+The+Game+Tommy+Edwards&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: Rock-in Robin Bobby Day
❌ Error retrieving valence for Rock-in Robin by Bobby Day: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Rock-in+Robin+Bobby+Day&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: Bird Dog The Everly Brothers


ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Bird Dog The Everly Brothers', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Topsy II Cozy Cole', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Little Star The Elegants', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': "It's Only Make Believe Conway Twitty", 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spot

❌ Error retrieving valence for Bird Dog by The Everly Brothers: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Bird+Dog+The+Everly+Brothers&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: Topsy II Cozy Cole
❌ Error retrieving valence for Topsy II by Cozy Cole: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Topsy+II+Cozy+Cole&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: Little Star The Elegants
❌ Error retrieving valence for Little Star by The Elegants: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Little+Star+The+Elegants&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: It's Only Make Believe Conway Twitty
❌ Error retrieving valence for It's Only Make Believe by Conway Twitty: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=It%27s+Only+Make+Believe+Conwa

ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': "Susie Darlin' Robin Luke", 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Nel Blu Dipinto Di Blu (Volaré) Domenico Modugno', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'The End Earl Grant', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Tom Dooley The Kingston Trio', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supporte

...

ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Return To Me Dean Martin', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Count Every Star The Rivieras', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Happy Years The Diamonds', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'When Will I Know George Hamilton IV', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERRO

❌ Error retrieving valence for Return To Me by Dean Martin: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Return+To+Me+Dean+Martin&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: Count Every Star The Rivieras
❌ Error retrieving valence for Count Every Star by The Rivieras: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Count+Every+Star+The+Rivieras&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: Happy Years The Diamonds
❌ Error retrieving valence for Happy Years by The Diamonds: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Happy+Years+The+Diamonds&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: When Will I Know George Hamilton IV
❌ Error retrieving valence for When Will I Know by George Hamilton IV: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=When+Wi

ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Try Me James Brown And The Famous Flames', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'Baubles Bangles And Beads The Kirby Stone Four', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported
ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': "Let's Go Steady For The Summer The Three G's", 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 400 due to Only valid bearer authentication supported


❌ Error retrieving valence for Try Me by James Brown And The Famous Flames: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Try+Me+James+Brown+And+The+Famous+Flames&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: Baubles Bangles And Beads The Kirby Stone Four
❌ Error retrieving valence for Baubles Bangles And Beads by The Kirby Stone Four: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Baubles+Bangles+And+Beads+The+Kirby+Stone+Four&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: Let's Go Steady For The Summer The Three G's
❌ Error retrieving valence for Let's Go Steady For The Summer by The Three G's: http status: 400, code: -1 - https://api.spotify.com/v1/search?q=Let%27s+Go+Steady+For+The+Summer+The+Three+G%27s&limit=1&offset=0&type=track:
 Only valid bearer authentication supported, reason: None
Searching for: City Lights Ray Price


KeyboardInterrupt: 

In [7]:
import spotipy
import requests
import pandas as pd
import time
from spotipy.oauth2 import SpotifyClientCredentials
import base64

# Your CSV file path
file_path = "/Users/inqsoncharoen/Documents/RStudio Module/yearly_billboard_chart.csv"
df = pd.read_csv(file_path)

# Your Spotify API credentials
SPOTIPY_CLIENT_ID = "572eb4f3685b49349cbd2517ac8c064e"
SPOTIPY_CLIENT_SECRET = "0e6ba40bf0e74257a939dabfb2019b9d"

# Manually obtain an access token
def get_access_token(client_id, client_secret):
    url = "https://accounts.spotify.com/api/token"
    headers = {
        "Authorization": "Basic " + base64.b64encode(f"{client_id}:{client_secret}".encode()).decode(),
        "Content-Type": "application/x-www-form-urlencoded",
    }
    data = {"grant_type": "client_credentials"}

    response = requests.post(url, headers=headers, data=data)
    
    if response.status_code == 200:
        return response.json().get("access_token")
    else:
        print(f"❌ Error: {response.status_code}, Response: {response.json()}")
        return None

# Get token
access_token = get_access_token(SPOTIPY_CLIENT_ID, SPOTIPY_CLIENT_SECRET)

# Initialize Spotipy client with manually set token
sp = spotipy.Spotify(auth=access_token)

# Function to get valence
# def get_valence(song, artist):
#     try:
#         query = f"{song} {artist}"
#         results = sp.search(q=query, type="track", limit=1)

#         if results['tracks']['items']:
#             track_id = results['tracks']['items'][0]['id']
#             features = sp.audio_features(track_id)

#             if features and features[0]:  # Check if features exist
#                 return features[0].get('valence', None)
#             else:
#                 print(f"⚠️ No valence data for {song} by {artist}")
#                 return None
#     except Exception as e:
#         print(f"❌ Error retrieving valence for {song} by {artist}: {e}")
#     return None

def get_valence(song, artist):
    try:
        query = f"{song} {artist}"
        results = sp.search(q=query, type="track", limit=1)

        if not results['tracks']['items']:  # If no results found, skip
            print(f"⚠️ Skipping: No results for {song} by {artist}")
            return None

        track_id = results['tracks']['items'][0]['id']
        features = sp.audio_features(track_id)

        if features and features[0]:  # Check if valence exists
            return features[0].get('valence', None)
        else:
            print(f"⚠️ Skipping: No valence data for {song} by {artist}")
            return None
    except Exception as e:
        print(f"❌ Error retrieving valence for {song} by {artist}: {e}")
        return None


# Apply valence function to each row in DataFrame
df['valence'] = df.apply(lambda row: get_valence(row['song'], row['artist']), axis=1)

# Save updated CSV
output_file = "yearly_billboard_chart_with_valence.csv"
df.to_csv(output_file, index=False)

print(f"✅ Updated CSV saved as {output_file}")


HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=07GtDOCxmye5KDWsTSACPk with Params: {} returned 403 due to None
HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=2tvt5K7y1gndmCgtIoLo1f with Params: {} returned 403 due to None


❌ Error retrieving valence for Chantilly Lace by Big Bopper: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=07GtDOCxmye5KDWsTSACPk:
 None, reason: None
❌ Error retrieving valence for It's All In The Game by Tommy Edwards: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=2tvt5K7y1gndmCgtIoLo1f:
 None, reason: None


HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=5sdZEmlwDJpISfCjyCneHi with Params: {} returned 403 due to None
HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=2ZCkqAo0tzzCVOth7ityh5 with Params: {} returned 403 due to None


❌ Error retrieving valence for Rock-in Robin by Bobby Day: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=5sdZEmlwDJpISfCjyCneHi:
 None, reason: None
❌ Error retrieving valence for Bird Dog by The Everly Brothers: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=2ZCkqAo0tzzCVOth7ityh5:
 None, reason: None


HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=3KVII1yBwfwuNQLZXma0oS with Params: {} returned 403 due to None
HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=67eLX8pb0aEvXWGQTpfRzf with Params: {} returned 403 due to None


...

ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=3zFwGvu3pZ6gquRS07LZSZ with Params: {} returned 403 due to None


❌ Error retrieving valence for The Worryin' Kind by Tommy Sands And The Raiders: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=3zFwGvu3pZ6gquRS07LZSZ:
 None, reason: None


ERROR:spotipy.client:HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=3XYoEYCdKEQLC9VK7mcrNQ with Params: {} returned 403 due to None


❌ Error retrieving valence for The Bird On My Head by David Seville: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=3XYoEYCdKEQLC9VK7mcrNQ:
 None, reason: None


KeyboardInterrupt: 